# Treinamento customizado de detecção de objetos · Ônibus & Chuva SP

Este notebook faz o **fine-tuning** (treinamento customizado) de uma rede YOLO para detectar, em pontos de ônibus:
`onibus`, `pessoa`, `guarda_chuva`, `cadeira_de_rodas` e `ponto_de_onibus`.

**Antes de começar**
1. Menu **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.
2. Tenha o dataset pronto (Roboflow, ou um `.zip` no Google Drive).
3. Rode as células **em ordem**. O treino leva de 20 a 40 minutos.

Tudo o que importa (pesos, gráficos, métricas) é salvo no **seu Google Drive**, porque o Colab apaga os arquivos quando a sessão acaba.

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install ultralytics roboflow

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
PASTA_DRIVE = '/content/drive/MyDrive/visao_onibus_chuva'
os.makedirs(PASTA_DRIVE, exist_ok=True)
print('Pasta de trabalho no Drive:', PASTA_DRIVE)

## Código do projeto
Troque o link pelo do repositório da equipe. Se ainda não criou o repositório, envie a pasta `src/` e `scripts/` pelo painel de arquivos do Colab (ícone de pasta à esquerda) para `/content/repo`.

In [ ]:
REPO = 'https://github.com/SEU-USUARIO/SEU-REPOSITORIO.git'
!rm -rf /content/repo
!git clone -q $REPO /content/repo
%cd /content/repo

## Passo 1 · Dataset
Use **uma** das duas opções.

**Opção A · Roboflow.** No projeto, vá em *Versions > Export Dataset > formato YOLOv8 > Show download code* e copie o nome do workspace, do projeto e a versão. A chave (API key) é pedida na hora e não fica salva no notebook.

In [ ]:
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass('Cole a API key do Roboflow (não aparece na tela): '))
projeto = rf.workspace('SEU_WORKSPACE').project('SEU_PROJETO')
dataset = projeto.version(1).download('yolov8', location='/content/dataset')

**Opção B · arquivo .zip no Drive.** O zip deve ter `data.yaml` e as pastas `train`, `valid` e `test` (cada uma com `images` e `labels`).

In [ ]:
!rm -rf /content/dataset && mkdir -p /content/dataset
!unzip -q "{PASTA_DRIVE}/dataset.zip" -d /content/dataset

### Ajustar o `data.yaml` e conferir o dataset

In [ ]:
import yaml, pathlib
DATASET = pathlib.Path('/content/dataset')
dados = yaml.safe_load(open(DATASET / 'data.yaml', encoding='utf-8'))
nomes = dados['names']
if isinstance(nomes, dict):
    nomes = [nomes[k] for k in sorted(nomes)]

def achar(*opcoes):
    for o in opcoes:
        if (DATASET / o).is_dir():
            return o

treino = achar('train/images', 'images/train')
val = achar('valid/images', 'val/images', 'images/val', 'images/valid')
teste = achar('test/images', 'images/test')
assert treino and val, 'Não achei as pastas de treino e validação. Confira a estrutura do dataset.'

novo = {'path': str(DATASET), 'train': treino, 'val': val, 'names': nomes}
if teste:
    novo['test'] = teste
yaml.safe_dump(novo, open(DATASET / 'data.yaml', 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
DATA_YAML = str(DATASET / 'data.yaml')
print(open(DATA_YAML, encoding='utf-8').read())

In [ ]:
!python /content/repo/scripts/verificar_dataset.py /content/dataset

**Guarde o resultado da célula acima**: é a tabela de imagens e instâncias por classe que vai no relatório.
Se alguma classe tem menos de 50 exemplos, junte mais imagens ou retire a classe antes de treinar.

## Passo 2 · Treinamento
O treino parte de pesos **pré-treinados** (*transfer learning*) e ajusta a rede às nossas classes.

In [ ]:
MODELO_BASE = 'yolo11n.pt'   # pequeno e rápido. Alternativas: 'yolo11s.pt' (mais preciso) ou 'yolov8n.pt'
EPOCAS = 60
TAMANHO = 640
LOTE = 16
PACIENCIA = 15               # para sozinho se não melhorar por 15 épocas
NOME_EXPERIMENTO = 'ponto_onibus'

In [ ]:
from ultralytics import YOLO
from pathlib import Path

modelo = YOLO(MODELO_BASE)
modelo.train(
    data=DATA_YAML, epochs=EPOCAS, imgsz=TAMANHO, batch=LOTE, patience=PACIENCIA,
    seed=42, project=f'{PASTA_DRIVE}/runs', name=NOME_EXPERIMENTO, plots=True,
)
PASTA_RUN = Path(modelo.trainer.save_dir)
print('Resultados em:', PASTA_RUN)

## Passo 3 · Gráficos do treino (prints para o relatório)

In [ ]:
from IPython.display import Image, display
for nome in ['results.png', 'confusion_matrix_normalized.png', 'BoxPR_curve.png', 'val_batch0_pred.jpg']:
    p = PASTA_RUN / nome
    if p.exists():
        print(nome)
        display(Image(filename=str(p), width=900))

## Passo 4 · Métricas do melhor modelo

In [ ]:
import pandas as pd
melhor = YOLO(PASTA_RUN / 'weights' / 'best.pt')

def avaliar(divisao):
    m = melhor.val(data=DATA_YAML, split=divisao, verbose=False)
    print(f'--- {divisao}: precisão {m.box.mp:.3f} | recall {m.box.mr:.3f} | mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}')
    linhas = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        linhas.append({'classe': melhor.names[int(c)], 'precisao': p, 'recall': r, 'mAP50': ap50, 'mAP50-95': ap})
    tabela = pd.DataFrame(linhas).round(3)
    display(tabela)
    tabela.to_csv(f'{PASTA_DRIVE}/metricas_{divisao}.csv', index=False)
    print('Velocidade (ms por imagem):', {k: round(v, 1) for k, v in m.speed.items()})

avaliar('val')
if (DATASET / 'test').is_dir() or (DATASET / 'images' / 'test').is_dir():
    avaliar('test')

## Passo 5 · Exemplos de detecção (imagens para o relatório)

In [ ]:
import glob, random
imgs = glob.glob('/content/dataset/test/images/*.*') or glob.glob('/content/dataset/valid/images/*.*')
amostra = random.Random(7).sample(imgs, min(6, len(imgs)))
preds = melhor.predict(amostra, conf=0.35, save=True, project=f'{PASTA_DRIVE}/exemplos', name='deteccoes', exist_ok=True, verbose=False)
for r in preds:
    display(Image(filename=str(Path(r.save_dir) / Path(r.path).name), width=700))

## Passo 6 · Guardar o modelo

In [ ]:
import shutil
shutil.copy(PASTA_RUN / 'weights' / 'best.pt', f'{PASTA_DRIVE}/best.pt')
print('Modelo salvo em', f'{PASTA_DRIVE}/best.pt')
# Baixe também para o seu computador, para rodar com a webcam:
from google.colab import files
files.download(f'{PASTA_DRIVE}/best.pt')

## Passo 7 · Teste em vídeo (mesmo código do tempo real)
Coloque um vídeo de ponto de ônibus em `visao_onibus_chuva/ponto.mp4` no Drive. O script processa quadro a quadro, como na webcam, só que sem abrir janela.

In [ ]:
!python /content/repo/src/detectar_tempo_real.py --pesos "{PASTA_DRIVE}/best.pt" --fonte "{PASTA_DRIVE}/ponto.mp4" --sem-janela --anonimizar --salvar-video /content/demo_raw.mp4 --csv /content/contagens.csv
!ffmpeg -y -loglevel error -i /content/demo_raw.mp4 -vcodec libx264 -pix_fmt yuv420p /content/demo.mp4
from IPython.display import Video
Video('/content/demo.mp4', embed=True, width=800)

In [ ]:
import pandas as pd
print(pd.read_csv('/content/contagens.csv').head(10).to_string())
shutil.copy('/content/contagens.csv', f'{PASTA_DRIVE}/contagens_exemplo.csv')
shutil.copy('/content/demo.mp4', f'{PASTA_DRIVE}/demo_colab.mp4')

## O que levar para o relatório
- Saída do `verificar_dataset.py` (tabela de imagens e instâncias por classe)
- `results.png` (curvas de treino) e `confusion_matrix_normalized.png`
- Tabela de métricas por classe e a velocidade em ms por imagem
- 3 a 6 imagens de exemplo com as detecções
- Trecho do CSV de contagens
- `best.pt` e o vídeo de teste